# Modeling code, deconstructed — a hands-on walkthrough

This notebook is a **teaching companion** to `notebooks/modeling.ipynb`. Where
that notebook *calls* the modeling functions as black boxes, this one **opens
them up**: for every core function in `peatfire.modeling` we run its body a few
lines at a time on *your* data, printing or displaying the intermediate object
each line produces — right down to the individual function calls inside a line.

By the end you will have watched a covariate get sampled at a pixel, a covariance
matrix get whitened into a Mahalanobis metric, a treated pixel get paired to its
nearest control, and a logistic coefficient become an odds ratio — each as a live
Python object you can poke at.

Sprinkled throughout are **understanding checks** — short quizzes that test whether
the concept landed before you move on.

## How to use it

1. **Run top to bottom.** Each section builds on live variables from the one
   before (`pixels`, `matched`, `frame`, …), exactly like the real pipeline.
2. **Read the markdown, then run the code under it.** Every code cell has a
   `🔎 This cell:` note saying exactly what it does; the surrounding markdown
   quotes the real source lines it re-runs.
3. **Do the quizzes.** A quiz has two cells: one shows the question (`q.ask()`),
   the next is where *you* answer (`q.answer("b")`). Running "all cells" will
   **not** spoil answers — you have to type a real choice.
4. **Use `src(fn)`** (defined below) any time you want to see a function's true,
   current source next to the deconstruction.

> This notebook **reads** your data and **fits** models but writes nothing to
> `data/` or `outputs/`. It is safe to re-run.


## 0 · Helpers: the quiz + inspection toolkit

Run this once. It defines:

* `Quiz` — a self-contained multiple-choice check (no `ipywidgets`, works in any
  Jupyter frontend). `q.ask()` prints the question; `q.answer("b")` grades it and
  explains. `q.answer("?")` (the placeholder) reveals nothing, so *Run All* never
  spoils the answers.
* `src(fn)` — pretty-print a function's real source, so the deconstruction always
  sits next to ground truth.
* `show(df)` — display a DataFrame's shape + head in one line.


🔎 **This cell:** defines the teaching tools used everywhere below — the `Quiz` class (multiple-choice checks), `src()` (print a function's real source), and `show()` (shape + head of a table). Run it first.

In [ ]:
import inspect
import textwrap

class Quiz:
    """A tiny, dependency-free multiple-choice understanding check."""
    def __init__(self, prompt, options, correct, explain):
        self.prompt = textwrap.dedent(prompt).strip()
        self.options = options            # dict like {"a": "...", "b": "..."}
        self.correct = str(correct).lower()
        self.explain = textwrap.dedent(explain).strip()

    def ask(self):
        print(self.prompt + "\n")
        for k, v in self.options.items():
            print(f"   ({k})  {v}")
        print("\n→ Answer in the next cell:  q.answer(\"a\")")

    def answer(self, choice):
        choice = str(choice).strip().lower()
        if choice in ("", "?"):
            print("Pick an option first, then re-run this cell, e.g.  q.answer(\"b\")")
            return
        ok = (choice == self.correct)
        print(("✅ Correct" if ok else "❌ Not quite") + f" — you chose ({choice}).")
        if not ok:
            print(f"The intended answer is ({self.correct}).")
        print("\n" + self.explain)


def src(fn, first=None):
    """Print a function's real source (optionally just the first N lines)."""
    lines = inspect.getsource(fn).splitlines()
    if first is not None:
        lines = lines[:first] + [f"    # ... ({len(inspect.getsource(fn).splitlines())-first} more lines)"]
    print("\n".join(lines))


def show(df, n=5, label=None):
    """Display shape + head of a (Geo)DataFrame."""
    if label:
        print(label)
    try:
        print("shape:", df.shape)
    except Exception:
        pass
    from IPython.display import display
    display(df.head(n))

print("helpers ready: Quiz, src(), show()")

## 1 · Imports — the public API *and* the private internals

The real notebook imports only the public functions. Here we **also** reach in
for the private helpers (`build_candidate_pool`, `pixelate`, `_stack_across_years`,
`standardized_mean_diff`, `_design_matrix`, the DiD stages, `_rasterize_values`,
`_prepare_design`, …) so we can run each stage in isolation.


🔎 **This cell:** imports the public modeling API *plus* the private/underscore helpers we deconstruct later, applies the project plot style, and sets the three global knobs (`RES_M`, `YEARS`, `restoration_yr_col`). Prints the analysis CRS and which covariate rasters are currently on disk.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from peatfire import (
    data_path, build_common_grid, to_common_grid, load_standardized, set_fire_style,
)
from peatfire.modeling import (
    load_completed_restoration_sites_in_analysis_crs,
    available_covariates, available_temporal_covariates, list_covariates,
    covariate_on_grid,
    attach_fire_response, build_frame, fit_logit_clustered, odds_ratios,
    climate,
    add_prognostic_score_series, add_propensity_score_series,
    match_controls_event_time, restrict_panel_to_matched,
)
from peatfire.modeling import did
from peatfire.modeling.matching import (
    get_treated_and_control_pixels, attach_covariates, match_controls,
    balance_table, plot_balance, assemble_units, check_matches, check_balance,
    # --- private internals we will deconstruct ---
    build_candidate_pool, pixelate, _stack_across_years, standardized_mean_diff,
    _design_matrix, _burned_wide, ANALYSIS_CRS,
)
from peatfire.modeling.did import (
    attach_cohort, build_panel, estimate_att, aggregate_att, NEVER_TREATED,
)
from peatfire.modeling.frame import _rasterize_values
from peatfire.modeling.models import _formula, _prepare_design
from peatfire.modeling.covariates import temporal_covariate_on_grid

set_fire_style()

RES_M = 300                 # analysis resolution (~FireCCIS311 native)
YEARS = range(2019, 2025)   # panel / fire-product coverage
restoration_yr_col = "End_Yr"

print("ANALYSIS_CRS =", ANALYSIS_CRS)
print("covariates on disk:", available_covariates())

## 2 · Load the treatment and the peat frame

Two inputs seed everything downstream:

* **Treatment** = completed restoration polygons (the canal-block sites).
* **Frame** = the 80%-histosol peat extent — the universe of pixels we sample.

`load_completed_restoration_sites_in_analysis_crs` is short but encodes a
**design decision**. Its body is essentially:

```python
gdf = load_restoration_sites_in_analysis_crs(path)          # read + reproject to EPSG:5070
gdf[restoration_yr_col] = gdf[restoration_yr_col].replace(0, np.nan)  # 0 == "no end year yet"
gdf = gdf.dropna(subset=restoration_yr_col)                 # keep only truly completed sites
```


🔎 **This cell:** loads the two seed layers — the completed restoration sites (treatment, already reprojected to EPSG:5070 with placeholder years dropped) and the 80%-histosol peat AOI (the sample frame) — and prints the surviving `End_Yr` values so you can see the treatment cohorts.

In [ ]:
peat_restoration = load_completed_restoration_sites_in_analysis_crs(
    restoration_yr_col=restoration_yr_col
)
aoi_nc_peat_80_histosol = gpd.read_file(
    data_path("processed", "peat_extent", "nc_peatlands_80_histosol_aoi.gpkg")
)

print("restoration CRS:", peat_restoration.crs, "| n sites:", len(peat_restoration))
print("End_Yr values:", sorted(peat_restoration[restoration_yr_col].unique()))
show(peat_restoration[[restoration_yr_col, "geometry"]], label="treatment (completed sites):")
show(aoi_nc_peat_80_histosol, label="peat frame (80% histosol AOI):")

### Understanding check

🔎 **This cell:** poses the quiz question about dropping `End_Yr == 0` sites (run it, read the options).

In [ ]:
q = Quiz(
    prompt="""Why does the loader `.replace(0, np.nan)` on End_Yr and then drop those rows?""",
    options={
        "a": "0 is a valid restoration year that we exclude for confidentiality",
        "b": "0 is a placeholder for 'no end year recorded yet'; without a real end year a "
             "site has no event time to anchor a before/after comparison, so it can't be treated",
        "c": "It shrinks the dataset so the notebook runs faster",
    },
    correct="b",
    explain="""
        The restoration YEAR is the moment a site flips from unrestored to restored
        (the DiD cohort `g` and the per-year `treated` flag both key off it). A 0
        means the end year is unknown, so those sites can't anchor an event time and
        are dropped from the treated group. Everything downstream can therefore assume
        every treated site has a real restoration year.
    """,
)
q.ask()

🔎 **This cell:** your answer slot — replace `?` with your choice to get graded and see the explanation.

In [ ]:
q.answer("?")   # <- replace ? with your choice, e.g. q.answer("b")

## 3 · Deconstruct `get_treated_and_control_pixels`

This orchestrator turns two polygon layers into a tidy **pixel-year panel**. It
chains three internal steps we'll run one at a time:

1. `build_candidate_pool` — clip-to-drop the treated sites (+ spillover halo)
   from the peat frame → the control candidate pool.
2. `pixelate` — lay a common grid, take cell centroids inside each polygon set.
3. `_stack_across_years` + event-time bookkeeping — repeat each pixel per year,
   compute `years_after_restoration` and the per-year `treated` flag.

First, look at the real thing:


🔎 **This cell:** prints the real first ~40 lines of `get_treated_and_control_pixels` so you can read the orchestration we're about to reproduce step by step.

In [ ]:
src(get_treated_and_control_pixels, first=40)

### 3a · `build_candidate_pool` — subtract shapes, don't keep them

The one idea here is `how="difference"`, the **inverse** of a clip: keep the peat
that is *outside* the restoration sites buffered by `spillover_m`.

```python
exclusion  = gpd.GeoDataFrame(geometry=[treated.buffer(spillover_m).union_all()], crs=...)
candidates = gpd.overlay(peat_aoi, exclusion, how="difference")
```


🔎 **This cell:** runs `build_candidate_pool`'s body by hand — buffers the treated sites, unions them into an exclusion mask, and `overlay(..., how="difference")` cuts them out of the peat frame — then confirms the area matches the real helper.

In [ ]:
spillover_m = 0   # TNC prevents spillover into neighbours, so no halo needed here

# --- run the body of build_candidate_pool line by line ---
treated_ = peat_restoration.to_crs(ANALYSIS_CRS)
peat_    = aoi_nc_peat_80_histosol.to_crs(ANALYSIS_CRS)

exclusion = gpd.GeoDataFrame(
    geometry=[treated_.buffer(spillover_m).union_all()], crs=treated_.crs
)
candidates = gpd.overlay(peat_, exclusion, how="difference")

print("peat frame area   (km^2):", round(peat_.area.sum()   / 1e6, 1))
print("candidate area    (km^2):", round(candidates.area.sum() / 1e6, 1))
print("removed (sites+halo):     ", round((peat_.area.sum() - candidates.area.sum()) / 1e6, 1))

# sanity: identical to the library helper
lib = build_candidate_pool(aoi_nc_peat_80_histosol, peat_restoration, spillover_m)
print("matches library helper area:",
      np.isclose(candidates.area.sum(), lib.area.sum(), rtol=1e-6))

### Understanding check

🔎 **This cell:** quiz on what `overlay(how="difference")` produces.

In [ ]:
q = Quiz(
    prompt="`gpd.overlay(peat_aoi, exclusion, how=\"difference\")` returns…",
    options={
        "a": "only the peat that overlaps the restoration buffer",
        "b": "the peat with the restoration sites (+ spillover halo) cut out — the control candidate pool",
        "c": "the restoration polygons themselves",
    },
    correct="b",
    explain="""
        `how="difference"` is the inverse of a clip: it keeps geometry A *minus* B.
        Here A is the peat frame and B is the treated sites buffered by spillover_m,
        so what's left is peat that is neither treated nor close enough to be
        partially rewetted by a neighbouring site — the pool controls are drawn from.
    """,
)
q.ask()

🔎 **This cell:** your answer slot for the candidate-pool quiz.

In [ ]:
q.answer("?")

### 3b · `pixelate` — polygons → grid-cell centroid points

Fire is a raster product, so we compare on a shared grid. `pixelate` builds a
common grid over the polygons, makes a point at every cell centre, and keeps the
points that fall **inside** the polygons (a spatial `within` join). The `carry`
argument copies polygon attributes (the site's `End_Yr`, `Proj_Name`) onto each
pixel so a treated pixel *remembers which site — and which restoration year — it
belongs to*.


🔎 **This cell:** reproduces `pixelate` for the treated polygons: build the shared grid, turn every grid cell centre into a point (`meshgrid` → `ravel`), then keep the points that fall `within` a restoration polygon — carrying `End_Yr`/`Proj_Name` onto each.

In [ ]:
# --- run pixelate's body for the TREATED polygons, carrying site attributes ---
carry = [restoration_yr_col, "Proj_Name"]
polygons = peat_restoration.to_crs(ANALYSIS_CRS)

grid = build_common_grid(peat_, RES_M, ANALYSIS_CRS)   # one grid shared by treated + control
print("grid shape (rows, cols):", (grid.rio.height, grid.rio.width))

xs, ys = grid["x"].values, grid["y"].values            # 1-D cell-centre coords
xx, yy = np.meshgrid(xs, ys)                            # 2-D fields
xx, yy = xx.ravel(), yy.ravel()                         # flattened to 1-D
print("candidate grid points:", xx.size)

points = gpd.GeoDataFrame(
    {"x": xx, "y": yy}, geometry=gpd.points_from_xy(xx, yy), crs=grid.rio.crs
)
inside = gpd.sjoin(points, polygons[["geometry", *carry]], predicate="within")
treated_pixels = inside[["x", "y", "geometry", *carry]].reset_index(drop=True)

print("treated pixels (inside restoration polygons):", len(treated_pixels))
show(treated_pixels)

### Understanding check

🔎 **This cell:** quiz on why site attributes are carried onto each pixel.

In [ ]:
q = Quiz(
    prompt="""Why does `pixelate` `carry=[End_Yr, Proj_Name]` onto each treated pixel,
              instead of just returning bare x/y points?""",
    options={
        "a": "So plots can colour pixels by site",
        "b": "So each treated pixel remembers its site's restoration year — which the "
             "per-year `treated` flag and the DiD cohort `g` are computed from",
        "c": "It's required by GeoPandas' sjoin",
    },
    correct="b",
    explain="""
        Treatment is an event in time. To know whether a pixel is 'treated' in 2022 you
        must know its site's restoration year; to place it in a DiD cohort you need that
        same year. Carrying End_Yr (and the site name for the cluster key) onto every
        pixel at pixelation time is what makes the per-year panel below possible.
    """,
)
q.ask()

🔎 **This cell:** your answer slot for the carry-through quiz.

In [ ]:
q.answer("?")

### 3c · Stack across years + event time

The geometry doesn't change year to year, so we pixelate **once** and cheaply
broadcast across `YEARS` with `_stack_across_years`. Then the two event-time
lines that define treatment:

```python
treated_panel["years_after_restoration"] = year - End_Yr
treated_panel["treated"] = (years_after_restoration >= 0).astype(int)
```

A restoration-site pixel is a **not-yet-treated control** (`treated == 0`) in
years *before* its site's `End_Yr`, and treated (`== 1`) from `End_Yr` onward.
Control-pool pixels are `treated == 0` in every year with a null `End_Yr`.


🔎 **This cell:** broadcasts the treated pixels across `YEARS` and computes the event-time columns: `years_after_restoration = year − End_Yr` and `treated = (that ≥ 0)`. The per-year counts show sites switching on as their restoration year arrives.

In [ ]:
# de-dup a pixel that lands in two overlapping sites -> keep earliest restoration year
treated_pts = (treated_pixels.sort_values(restoration_yr_col)
               .drop_duplicates(subset=["x", "y"], keep="first").reset_index(drop=True))

treated_panel = _stack_across_years(treated_pts, list(YEARS))
treated_panel[restoration_yr_col] = treated_panel[restoration_yr_col].astype("float64")
treated_panel["years_after_restoration"] = treated_panel["year"] - treated_panel[restoration_yr_col]
treated_panel["treated"] = (treated_panel["years_after_restoration"] >= 0).astype(int)

print("treated pixel-years per calendar year (1 = restored by then):")
print(treated_panel.groupby("year")["treated"].agg(["sum", "count"]))
show(treated_panel[["x", "y", "year", restoration_yr_col,
                    "years_after_restoration", "treated"]].head(8),
     n=8, label="event-time bookkeeping for a few treated pixels:")

Now let the real orchestrator build the full panel (treated + control) —
it does exactly the above plus the control side. We'll use `pixels` from here on.

🔎 **This cell:** calls the real `get_treated_and_control_pixels` to build the full treated+control pixel-year panel `pixels` (used by every later section) and shows its per-year size.

In [ ]:
pixels = get_treated_and_control_pixels(
    aoi_nc_peat_80_histosol, peat_restoration,
    years=YEARS, spillover_m=spillover_m, res_m=RES_M,
    treated_col_name="treated",
    restoration_yr_col=restoration_yr_col, site_col="Proj_Name",
)
print("panel shape:", pixels.shape)
print("rows per calendar year:")
print(pixels.groupby("year")["treated"].agg(n_pixels="count", n_treated="sum"))
show(pixels)

### Understanding check

🔎 **This cell:** quiz on why pre-restoration site pixels are kept as not-yet-treated controls.

In [ ]:
q = Quiz(
    prompt="""A site has End_Yr = 2022. In the 2020 panel row, that site's pixel has
              treated = 0. Why keep it at all rather than dropping pre-restoration rows?""",
    options={
        "a": "It's a bug; those rows should be dropped",
        "b": "Because in 2020 the site is genuinely not-yet-treated, and the staggered "
             "DiD uses not-yet-treated units as valid controls for other cohorts",
        "c": "To pad the panel so every year has equal rows",
    },
    correct="b",
    explain="""
        In a staggered design, a unit that hasn't been treated *yet* is a legitimate
        control for units treated earlier. Dropping those rows (drop_pretreatment=True)
        gives you the simpler restored-by-year cross-section, but throws away the
        comparisons the Callaway-Sant'Anna estimator relies on. Keeping them (the
        default) is what makes the event study and its parallel-trends check possible.
    """,
)
q.ask()

🔎 **This cell:** your answer slot for the not-yet-treated quiz.

In [ ]:
q.answer("?")

## 4 · Deconstruct `attach_covariates` — sample rasters at pixels

Each covariate lives as a raster. `attach_covariates` reads the raster value at
every pixel. The clever bit: a **static** covariate (elevation, soil, climate
normal) depends only on *where* a pixel is, not on the year, so it samples each
layer **once on the unique `(x, y)` pixels** and broadcasts back across the years
— not once per pixel-year. Per-year (temporal) layers are sampled per year and
joined on `(x, y, year)`.

Pick the continuous covariates that are actually on disk (these become the match
axes later); categoricals become exact-match keys.


🔎 **This cell:** chooses which covariates the rest of the notebook uses: the continuous layers on disk become the matching axes; the categorical layers become exact-match keys.

In [ ]:
on_disk = set(available_covariates())
covariates  = [c for c in list_covariates("continuous")  if c in on_disk] or ["histosol_pct", "elevation"]
categorical = [c for c in list_covariates("categorical") if c in on_disk]
print("continuous (match axes):", covariates)
print("categorical (exact-match keys):", categorical or "(none on disk)")

🔎 **This cell:** runs the static half of `attach_covariates`: collapse to the distinct `(x, y)` pixels, warp each covariate raster onto the grid, and read its value at every pixel with `sel(method="nearest")` — one read per layer, not per pixel-year.

In [ ]:
# --- attach_covariates body, static part, on live data ---
grid = build_common_grid(aoi_nc_peat_80_histosol, RES_M, ANALYSIS_CRS)

unique_px = pixels[["x", "y"]].drop_duplicates().reset_index(drop=True)
print("distinct pixels:", len(unique_px), "  vs pixel-year rows:", len(pixels),
      f"  ({len(pixels)//max(len(unique_px),1)}x fewer raster reads this way)")

xi = xr.DataArray(unique_px["x"].values, dims="point")
yi = xr.DataArray(unique_px["y"].values, dims="point")
for name in covariates:
    cov = covariate_on_grid(name, grid, aoi_nc_peat_80_histosol)
    if cov is None:
        print("  (skip, not on disk):", name); continue
    unique_px[name] = cov.sel(x=xi, y=yi, method="nearest").values
    v = unique_px[name]
    print(f"  {name:<22} sampled  min={v.min():.3g}  median={v.median():.3g}  max={v.max():.3g}")

show(unique_px, label="one row per physical pixel, covariates sampled once:")

Now the broadcast-back merge — the same value repeats across a pixel's
years — and a check that it equals the library call.

🔎 **This cell:** merges the per-pixel values back onto every pixel-year row (the broadcast), then calls the real `attach_covariates` and verifies the hand-built column matches — producing `pixels_with_covariates`, the input to matching.

In [ ]:
sampled = [c for c in unique_px.columns if c not in ("x", "y")]
by_hand = pixels.merge(unique_px[["x", "y", *sampled]], on=["x", "y"], how="left")

pixels_with_covariates = attach_covariates(
    pixels, [*covariates, *categorical], aoi_nc_peat_80_histosol, RES_M
)
print("hand merge shape:", by_hand.shape, "| library shape:", pixels_with_covariates.shape)
print("elevation identical:",
      np.allclose(by_hand["elevation"].to_numpy(),
                  pixels_with_covariates["elevation"].to_numpy(), equal_nan=True)
      if "elevation" in covariates else "n/a")
show(pixels_with_covariates)

### Understanding check

🔎 **This cell:** quiz on the sample-once-then-broadcast optimization.

In [ ]:
q = Quiz(
    prompt="""`attach_covariates` calls `pixels[["x","y"]].drop_duplicates()` and samples
              the raster on THOSE rows, then merges back. Why not sample straight on the
              full pixel-year panel?""",
    options={
        "a": "drop_duplicates is required to build an xarray selector",
        "b": "A static covariate is identical across a pixel's years, so sampling per "
             "pixel-year would repeat the same raster read N times for no new information",
        "c": "Because the panel rows are in the wrong order",
    },
    correct="b",
    explain="""
        Elevation in 2019 == elevation in 2024 for a fixed pixel. Sampling once per
        distinct (x, y) and broadcasting gives an identical result with ~N-fewer raster
        reads (N = number of years). Temporal layers (weather) are the exception — those
        genuinely differ by year and are sampled per year, joined on (x, y, year).
    """,
)
q.ask()

🔎 **This cell:** your answer slot for the broadcast quiz.

In [ ]:
q.answer("?")

## 5 · Deconstruct `attach_fire_response` — the 0/1 outcome

The response is *swappable*: `load_standardized(product, year, aoi)` returns a
burned raster for a given product/year. `attach_fire_response` warps it onto the
grid (`how="max"`, so any burned sub-cell lights the cell) and reads it at each
pixel-year point. Years the product doesn't cover stay `NaN`.


🔎 **This cell:** prints the head of `attach_fire_response` so you can read its per-year sampling loop.

In [ ]:
src(attach_fire_response, first=6)

🔎 **This cell:** runs `attach_fire_response`'s body: for each year, load the FireCCIS311 burned raster, warp it onto the grid with `how="max"`, and read 0/1 at each pixel. Then `did.prepare_panel` adds DiD bookkeeping, giving the `panel` used in §6 and §8.

In [ ]:
# --- attach_fire_response body on live data ---
out = pixels_with_covariates.copy()
out["burned"] = np.nan
grid = build_common_grid(aoi_nc_peat_80_histosol, res_m=RES_M)

for year, idx in out.groupby("year").groups.items():
    resp = load_standardized("FireCCIS311", int(year), aoi_nc_peat_80_histosol)
    if resp is None:
        print(f"  {year}: no coverage -> NaN"); continue
    burned = to_common_grid(resp.astype("float32"), grid, how="max")
    sub = out.loc[idx]
    xi = xr.DataArray(sub["x"].values, dims="point")
    yi = xr.DataArray(sub["y"].values, dims="point")
    out.loc[idx, "burned"] = burned.sel(x=xi, y=yi, method="nearest").values
    print(f"  {year}: burned pixel-years = {int(np.nansum(out.loc[idx, 'burned']))}")

panel = did.prepare_panel(out, restoration_yr_col=restoration_yr_col, site_col="Proj_Name")
print("\nburned rate by treated (panel):")
print(panel.groupby("treated")["burned"].mean())

## 6 · Deconstruct the staggered DiD (`prepare_panel` → `fit_att`)

`did.prepare_panel` adds the two things the Callaway–Sant'Anna estimator needs:

1. a stable **`entity`** id per distinct pixel (`groupby([x,y]).ngroup()`), and
2. the **cohort `g`** — each site's first-treatment year, `0` for never-treated —
   via `attach_cohort`.

Let's rebuild both from `out` (the response panel) and inspect the cohorts.


🔎 **This cell:** reproduces `prepare_panel`: assign each distinct pixel an `entity` id, build the site → restoration-year map, and `attach_cohort` to stamp the cohort `g` (0 = never-treated). Prints the cohorts and the entity count.

In [ ]:
# --- prepare_panel body ---
p = out.copy()
p["entity"] = p.groupby(["x", "y"]).ngroup()                 # (1) panel entity id
cohort_by = (p.dropna(subset=[restoration_yr_col])
               .groupby("Proj_Name")[restoration_yr_col].first())
print("site -> first restoration year (cohort source):")
print(cohort_by)

p = attach_cohort(p, cohort_by=cohort_by, key="Proj_Name", treated_col="treated")
print("\ncohorts g present:", sorted(p['g'].unique()), " (0 = never-treated control)")
print("distinct entities (pixels):", p['entity'].nunique())

### Understanding check

🔎 **This cell:** quiz on the never-treated cohort sentinel.

In [ ]:
q = Quiz(
    prompt="In this codebase, a never-treated control pixel gets which cohort value g?",
    options={"a": "NaN", "b": "-1", "c": "0 (the NEVER_TREATED sentinel)"},
    correct="c",
    explain=f"""
        `NEVER_TREATED = {NEVER_TREATED}`. Treated sites get g = their restoration year;
        controls (and any unmapped stratum) fall through to 0. `build_panel` then insists
        BOTH exist — treated cohorts (g>0) AND never/not-yet-treated (g==0) — because
        with no comparison group there is nothing to difference against.
    """,
)
q.ask()

🔎 **This cell:** your answer slot for the cohort quiz.

In [ ]:
q.answer("?")

Now the estimation chain that `fit_att` wraps:
`build_panel` (validate + index by `(entity, year)`) → `estimate_att`
(doubly-robust Callaway–Sant'Anna) → `aggregate_att` ("simple" overall + "event"
study). The estimator needs the optional `differences` backend, so we guard it —
exactly like the real notebook.

🔎 **This cell:** runs the three DiD stages `fit_att` chains — `build_panel` → `estimate_att` (doubly-robust ATT) → `aggregate_att` (overall + event study) — inside a try/except so a missing `differences` backend doesn't halt the notebook. Displays the event study.

In [ ]:
covs = [c for c in covariates if c in panel.columns]
event_study = overall = att = None
try:
    # deconstructed: build the panel, then estimate, then aggregate
    bp = build_panel(panel, entity="unit_id", time="year", response="burned", covariates=covs)
    print("balanced panel indexed by (entity, year):", bp.shape)
    att     = estimate_att(bp, response="burned", covariates=covs, est_method="dr")
    overall = aggregate_att(att, kind="simple")
    event_study = aggregate_att(att, kind="event")
    print("overall ATT (Castro's headline):", overall)
except Exception as e:
    print("staggered DiD skipped (needs `differences` backend + fire coverage):", repr(e))

event_study

### Understanding check

🔎 **This cell:** quiz on reading the pre-treatment event-study points as the parallel-trends check.

In [ ]:
q = Quiz(
    prompt="""In the event study, the PRE-treatment points (event time < 0) are used as…""",
    options={
        "a": "the treatment effect itself",
        "b": "the parallel-trends check — they should straddle zero, i.e. treated and "
             "control burned alike BEFORE restoration",
        "c": "placebo years to be discarded",
    },
    correct="b",
    explain="""
        DiD identifies off the CHANGE after treatment relative to controls. That's only
        credible if the two groups were on the same trajectory beforehand. The pre-period
        event-study coefficients estimate exactly that pre-trend; near-zero pre-points are
        the evidence for parallel trends. The post-period points then trace how the effect
        evolves after the canal blocks go in.
    """,
)
q.ask()

🔎 **This cell:** your answer slot for the parallel-trends quiz.

In [ ]:
q.answer("?")

## 7 · Deconstruct `match_controls` — the causal-design heart

This is the function most worth opening up. It pairs each treated pixel with its
nearest control on the covariates, using **Mahalanobis** distance, within a
**caliper**, optionally **exact-matched** on categoricals and geographically
capped. We'll walk its six internal steps on `pixels_with_covariates`.


🔎 **This cell:** prints the head of `match_controls` and its step-by-step rationale docstring, which the next cells reproduce one step at a time.

In [ ]:
src(match_controls, first=12)

### Step 1 — define the treated *group* (not the per-year flag)

Matching is on *static* covariates, so we collapse the pixel-year panel to one
row per physical pixel and label the group by restoration-site membership
(`End_Yr.notna()`), independent of the per-year `treated` flag.


🔎 **This cell:** Step 1 — collapses the panel to one row per physical pixel and labels the matching group by restoration-site membership (`End_Yr.notna()`), not the per-year flag.

In [ ]:
df = pixels_with_covariates.copy()
cross = df.drop_duplicates(subset=["x", "y"]).copy()
cross["_grp_treated"] = cross[restoration_yr_col].notna().astype(int)
print("distinct pixels:", len(cross))
print("group sizes  (1 = restoration-site pixel, 0 = candidate control):")
print(cross["_grp_treated"].value_counts())

### Step 2 — covariate matrix, drop pixels NaN in any covariate

🔎 **This cell:** Step 2 — drops pixels with any missing covariate and builds the numeric matrix `X`; the printed per-column means/SDs show the wildly different scales that motivate whitening.

In [ ]:
cont = list(covariates)
cross = cross.dropna(subset=cont).reset_index(drop=True)
X = cross[cont].to_numpy(dtype=float)
print("covariate matrix X:", X.shape, "  (n_pixels x n_covariates)")
print("means :", np.round(X.mean(axis=0), 3))
print("stdev :", np.round(X.std(axis=0),  3), " <- different scales -> must whiten")

### Step 3 — whiten so plain Euclidean **becomes** Mahalanobis

This is the mathematical core. We center `X`, form its covariance, and compute
`W = cov^{-1/2}` via a symmetric eigendecomposition. Then `Xw = Xc @ W`. In the
whitened space, ordinary Euclidean distance equals Mahalanobis distance in the
original covariate space — i.e. distances are **scale-free and decorrelated**.

**Predict first:** what should `np.cov(Xw, rowvar=False)` look like? Run and see.


🔎 **This cell:** Step 3 — the whitening math: center `X`, take its covariance, form `W = cov^{-1/2}` via eigendecomposition, and apply it. The whitened covariance printing as ~identity proves distances are now scale-free and decorrelated (= Mahalanobis).

In [ ]:
mean = X.mean(axis=0)
Xc = X - mean
cov = np.atleast_2d(np.cov(Xc, rowvar=False))
cov += 1e-6 * np.eye(cov.shape[0])              # ridge: keep it invertible
evals, evecs = np.linalg.eigh(cov)              # cov = V diag(w) V^T (symmetric)
evals = np.clip(evals, 1e-12, None)
W = evecs @ np.diag(1.0 / np.sqrt(evals)) @ evecs.T   # cov^{-1/2}
Xw = Xc @ W

print("original covariance (rounded):")
print(np.round(cov, 2))
print("\nwhitened covariance  ~ IDENTITY:")
print(np.round(np.cov(Xw, rowvar=False), 2))

Concretely, whitening changes *who is nearest*. Take one treated pixel and
compare its nearest control under raw-Euclidean vs whitened distance.

🔎 **This cell:** shows whitening actually changes the answer: for one treated pixel it finds the nearest control under raw Euclidean vs whitened (Mahalanobis) distance — often a different control, because raw distance is dominated by the large-scale covariate.

In [ ]:
zcols = [f"_z{i}" for i in range(Xw.shape[1])]
cross_w = cross.assign(**{c: Xw[:, i] for i, c in enumerate(zcols)})

t_mask = cross_w["_grp_treated"].eq(1).to_numpy()
c_mask = ~t_mask
t0 = np.where(t_mask)[0][0]                       # first treated pixel

d_raw = np.linalg.norm(X[c_mask] - X[t0],  axis=1)   # raw covariate units
d_wht = np.linalg.norm(Xw[c_mask] - Xw[t0], axis=1)  # whitened / Mahalanobis
print("nearest control by RAW  Euclidean: control #", int(np.argmin(d_raw)),
      " dist", round(d_raw.min(), 3))
print("nearest control by WHITENED (Maha): control #", int(np.argmin(d_wht)),
      " dist", round(d_wht.min(), 3))
print("same control?", int(np.argmin(d_raw)) == int(np.argmin(d_wht)))

### Understanding check

🔎 **This cell:** quiz on why we whiten before measuring distance.

In [ ]:
q = Quiz(
    prompt="""Why whiten the covariates before measuring distance, instead of plain
              Euclidean on the raw columns?""",
    options={
        "a": "Euclidean can't handle negative numbers",
        "b": "Raw Euclidean is dominated by whichever covariate has the largest numeric "
             "spread and double-counts correlated covariates; whitening puts every "
             "covariate on equal footing and decorrelates them (= Mahalanobis)",
        "c": "Whitening makes the code run faster",
    },
    correct="b",
    explain="""
        Elevation (~metres) and histosol % (0-100) live on different scales, so raw
        Euclidean distance is mostly 'whoever has the bigger units'. And correlated
        covariates (e.g. elevation & a climate normal) get counted twice. Multiplying by
        cov^{-1/2} rescales to unit variance AND removes the correlation, so distance in
        the whitened space is Mahalanobis distance in the original space — the honest
        'how many SDs apart, accounting for shared signal' metric.
    """,
)
q.ask()

🔎 **This cell:** your answer slot for the whitening quiz.

In [ ]:
q.answer("?")

### Steps 4–5 — exact-match groups, geographic cap, caliper, greedy pairing

With `categorical` given, matching happens **within** each class (`groupby`). For
each treated pixel: optionally restrict to controls within `max_dist_m` metres
(a KD-tree query — geography first), rank the eligible controls by whitened
distance, and take the nearest `k` inside the `caliper`, skipping already-used
controls when `replace=False`. A treated pixel with no control inside the caliper
is **dropped** rather than matched to a poor twin.

Rather than re-implement the whole bookkeeping, run the real function and inspect
its diagnostics + output — then read the annotated source.


🔎 **This cell:** runs the real `match_controls` (with its printed drop diagnostics) to produce `matched`, then confirms every control's `match_distance` sits inside the caliper.

In [ ]:
caliper, k = 1.0, 1
matched = match_controls(
    pixels_with_covariates,
    continuous=covariates, categorical=categorical,
    caliper=caliper, k=k, replace=False,
    restoration_yr_col=restoration_yr_col, site_col="Proj_Name",
)
print("\nmatched columns:", list(matched.columns))
print("rows by treated:", dict(matched["treated"].value_counts()))
print("max control match_distance:", round(matched.loc[matched.treated==0, "match_distance"].max(), 3),
      " (<= caliper", caliper, ")")
show(matched[["unit_id", "site_id", "pair_id", "treated", "match_distance", *covariates]])

🔎 **This cell:** prints the full `match_controls` source so you can trace the geography-first KD-tree filter, the caliper `break`, and the no-replacement bookkeeping end to end.

In [ ]:
# read the greedy pairing loop (steps 5a/5b) in full
src(match_controls)

### Understanding checks (matching)

🔎 **This cell:** quiz on the units of the caliper vs `max_dist_m`.

In [ ]:
q = Quiz(
    prompt="The `caliper` (default 1.0) and `max_dist_m` are ceilings in which units?",
    options={
        "a": "both in metres",
        "b": "caliper is in whitened / Mahalanobis SD units; max_dist_m is in metres (physical distance)",
        "c": "both in standard deviations",
    },
    correct="b",
    explain="""
        The caliper bounds COVARIATE distance in the whitened space, so '1.0' means
        'within ~1 Mahalanobis SD'. `max_dist_m` is a separate, PHYSICAL ceiling in
        metres (Castro's spatial restriction), applied geography-first via a KD-tree so a
        nearby, decent covariate match is never missed for falling outside a fixed
        neighbour budget. None (default) imposes no spatial restriction.
    """,
)
q.ask()

🔎 **This cell:** your answer slot for the caliper-units quiz.

In [ ]:
q.answer("?")

🔎 **This cell:** quiz on why `site_id` (the cluster key) matters downstream.

In [ ]:
q = Quiz(
    prompt="`site_id` on the matched output is the restoration SITE, and controls inherit "
           "their treated partner's site. Why does that matter downstream?",
    options={
        "a": "It's only for plotting colours",
        "b": "It's the CLUSTER key: standard errors later cluster on site because the "
             "effective sample size is the handful of restoration sites, not the thousands "
             "of correlated pixels within them",
        "c": "It lets build_frame sort rows",
    },
    correct="b",
    explain="""
        Pixels inside one restoration project are highly correlated — treating each as an
        independent observation massively overstates significance. Giving every treated
        pixel and its matched controls a shared site_id lets fit_logit_clustered cluster
        the SEs on the site, so the honest N is the number of sites.
    """,
)
q.ask()

🔎 **This cell:** your answer slot for the cluster-key quiz.

In [ ]:
q.answer("?")

### Step 6 — balance: does the match actually work? (`balance_table` + SMD)

The **standardized mean difference** per covariate, before vs after matching:
`SMD = (mean_treated − mean_control) / pooled_SD`. `|SMD| < 0.1` is the usual bar
for "balanced". Let's compute one SMD by hand, then the whole before/after table
and the love plot.


🔎 **This cell:** computes the standardized mean difference for one covariate by hand — (mean_t − mean_c) / pooled SD — and checks it against the library's `standardized_mean_diff`, so the balance metric is demystified.

In [ ]:
# SMD by hand for the first covariate, treated vs candidate-control, BEFORE matching
name = covariates[0]
uniq = pixels_with_covariates.drop_duplicates(subset=["x", "y"])
is_t = uniq[restoration_yr_col].notna()
t_vals = uniq.loc[is_t,  name].to_numpy(dtype=float)
c_vals = uniq.loc[~is_t, name].to_numpy(dtype=float)
t_vals, c_vals = t_vals[~np.isnan(t_vals)], c_vals[~np.isnan(c_vals)]
pooled = np.sqrt((t_vals.var(ddof=1) + c_vals.var(ddof=1)) / 2)
smd_by_hand = (t_vals.mean() - c_vals.mean()) / pooled
print(f"{name}: SMD by hand = {smd_by_hand:.3f}")
print(f"{name}: standardized_mean_diff() = {standardized_mean_diff(t_vals, c_vals):.3f}")

🔎 **This cell:** builds the before/after balance tables, runs the `check_matches`/`check_balance` assertions, and draws the love plot — the figure showing every covariate's |SMD| collapsing toward 0 after matching.

In [ ]:
before = balance_table(pixels_with_covariates, continuous=covariates,
                       restoration_yr_col=restoration_yr_col)
after  = balance_table(matched, continuous=covariates)
check_matches(matched, caliper=caliper)
check_balance(before, after)
print(before.join(after, lsuffix="_before", rsuffix="_after"))

ax = plot_balance(before, after)
plt.show()

### Understanding check

🔎 **This cell:** quiz on reading a |SMD| drop on the love plot.

In [ ]:
q = Quiz(
    prompt="On the love plot, a covariate's |SMD| moves from 0.8 (before) to 0.05 (after). That means…",
    options={
        "a": "matching made that covariate more imbalanced",
        "b": "before matching, treated and control differed by ~0.8 pooled SDs on that "
             "covariate; after matching they differ by ~0.05 SD — now well-balanced (< 0.1)",
        "c": "the covariate was dropped from the model",
    },
    correct="b",
    explain="""
        SMD is the gap between the two groups' means in pooled-SD units. Big before,
        tiny after = the match found controls that look like the treated pixels on that
        covariate. Getting every covariate's |SMD| under 0.1 is what later lets us
        attribute a fire difference to restoration rather than to geography.
    """,
)
q.ask()

🔎 **This cell:** your answer slot for the balance quiz.

In [ ]:
q.answer("?")

### `assemble_units` — package the matched pixels for `build_frame`

The last matching step keeps the required `[unit_id, site_id, treated, geometry]`
plus **the already-sampled covariate columns**. That last part matters: matched
pixels are zero-area centroid POINTs, so re-reading rasters at them later would
null everything — carrying the values through avoids that trap.


🔎 **This cell:** calls `assemble_units` to shape `matched` into the `units` GeoDataFrame `build_frame` consumes, confirming the sampled covariate columns ride along.

In [ ]:
units = assemble_units(matched)
print("units by treatment:", dict(units["treated"].value_counts()))
print("carried columns:", [c for c in units.columns if c in covariates])
show(units)

## 8 · Deconstruct the faithful-Castro scores (per-year prognostic + per-vintage propensity)

The §7 match collapses covariates into one distance. Castro instead matches on a
**time series of predicted risk**. Two score builders feed that:

* `add_prognostic_score_series` → `phat_<year>`: a *separate* logistic fire-risk
  model **per outcome year**, trained **only on never-treated controls**, then
  predicted for everyone. The per-year coefficients capture how baseline risk
  shifts across dry/wet years.
* `add_propensity_score_series` → `psm_<g>`: one logistic **per restoration
  vintage `g`** (pixels of vintage `g` = positives, never-treated = negatives),
  projected onto every pixel.

Let's watch a single prognostic year get fit from the inside.


🔎 **This cell:** opens up ONE year of `add_prognostic_score_series`: build the standardized design matrix and the pixel×year burn table, fit a logistic on never-treated controls only for a middle year, then predict that year's baseline fire risk `phat_<yr>` for every pixel.

In [ ]:
scored = panel.copy()
if not isinstance(scored, gpd.GeoDataFrame) or scored.geometry.isna().all():
    scored = gpd.GeoDataFrame(
        scored, geometry=gpd.points_from_xy(scored["x"], scored["y"]), crs="EPSG:5070")

# --- inside add_prognostic_score_series: fit ONE year on never-treated controls ---
from sklearn.linear_model import LogisticRegression

uniq = scored.drop_duplicates(subset=["x", "y"]).reset_index(drop=True)
never = uniq[restoration_yr_col].isna().to_numpy()           # never-treated controls
ok = uniq[covariates].notna().all(axis=1).to_numpy()
use = uniq.loc[ok].reset_index(drop=True)
never_use = never[ok]
Xd = _design_matrix(use, covariates, [])                     # standardized design matrix
bw = _burned_wide(scored, "burned", "year")                  # pixel x year 0/1 table

yr = sorted(bw.columns)[len(bw.columns) // 2]                # a middle year
tgt = use.merge(bw[yr].rename("_t").reset_index(), on=["x", "y"], how="left")["_t"].to_numpy()
train = never_use & ~np.isnan(tgt)
print(f"fitting prognostic for {yr}: training on {int(train.sum())} never-treated controls "
      f"with a defined outcome; {int(np.nansum(tgt[train]))} of them burned")
if len(np.unique(tgt[train].astype(int))) >= 2:
    model = LogisticRegression(max_iter=1000, C=1.0).fit(Xd[train], tgt[train].astype(int))
    phat = model.predict_proba(Xd)[:, 1]                     # predicted for EVERYONE
    print(f"phat_{yr}: min={phat.min():.4g}  mean={phat.mean():.4g}  max={phat.max():.4g}")
else:
    print("not enough fire variation that year to fit (would fall back to base rate)")

Now let the real builders attach the full series, then inspect the columns.

🔎 **This cell:** calls both real score builders to attach the full `phat_<year>` (per-year prognostic) and `psm_<g>` (per-vintage propensity) series onto `scored`, and lists the resulting columns.

In [ ]:
match_cat = [c for c in categorical if c in scored.columns]
scored = add_prognostic_score_series(scored, continuous=covariates, categorical=match_cat,
                                     response="burned", restoration_yr_col=restoration_yr_col)
scored = add_propensity_score_series(scored, continuous=covariates, categorical=match_cat,
                                     restoration_yr_col=restoration_yr_col)
phat_cols = sorted((c for c in scored.columns if c.startswith("phat_") and c[5:].isdigit()),
                   key=lambda c: int(c[5:]))
psm_cols  = sorted((c for c in scored.columns if c.startswith("psm_") and c[4:].isdigit()),
                   key=lambda c: int(c[4:]))
print("per-year prognostic  :", phat_cols)
print("per-vintage propensity:", psm_cols)
show(scored.drop_duplicates(["x", "y"])[["x", "y", restoration_yr_col, *phat_cols, *psm_cols]])

### Understanding check

🔎 **This cell:** quiz on why the prognostic model trains on never-treated controls (leakage).

In [ ]:
q = Quiz(
    prompt="""The prognostic model is trained ONLY on never-treated control pixels, then
              predicted for treated pixels too. Why train on controls only?""",
    options={
        "a": "There aren't enough treated pixels to train on",
        "b": "The prognostic score is BASELINE (untreated) risk. Training on never-treated "
             "controls means a treated pixel's score is a pure covariate prediction — it "
             "never sees that pixel's own post-restoration burns, so no outcome leakage",
        "c": "Controls are easier to fit numerically",
    },
    correct="b",
    explain="""
        Psi(X) = E[burn | X, untreated] is a pixel's fire risk ABSENT treatment. If you
        trained it on treated pixels' post-restoration outcomes, a treated pixel's score
        would partly encode the very effect you're trying to estimate — leakage. Fitting
        on never-treated controls and projecting keeps the score a clean function of the
        covariates only.
    """,
)
q.ask()

🔎 **This cell:** your answer slot for the prognostic-score quiz.

In [ ]:
q.answer("?")

### The event-time trajectory match + match-first DiD

`match_controls_event_time` assembles, **per vintage `g`**, a matching vector =
pre-construction fire lags (`g−1`, `g−2`, a drought benchmark) + forward
`phat_<t>` for `t ≥ g` + the vintage propensity `psm_<g>`, then runs the same
Mahalanobis/caliper matcher on that vector. `restrict_panel_to_matched` then keeps
only the matched pixels so the DiD runs against matched controls. Guarded for the
optional backend.


🔎 **This cell:** runs the faithful-Castro two-step: an event-time trajectory match (`match_controls_event_time`), then `restrict_panel_to_matched` + `fit_att` so the staggered DiD is identified against the matched controls only. Guarded for the optional backend.

In [ ]:
try:
    matched_et = match_controls_event_time(
        scored, response="burned", categorical=match_cat,
        restoration_yr_col=restoration_yr_col, site_col="Proj_Name",
        pre_lags=(1, 2), drought_year=2015, caliper=1.0, k=1, carry=covariates,
    )
    print("matched pairs:", matched_et["pair_id"].nunique(),
          "| cohorts:", sorted(matched_et["cohort"].unique()),
          "| site clusters:", matched_et["site_id"].nunique())

    panel_m = did.prepare_panel(restrict_panel_to_matched(scored, matched_et),
                                restoration_yr_col=restoration_yr_col, site_col="Proj_Name")
    covs_m = [c for c in covariates if c in panel_m.columns]
    att_m, overall_m, es_m = did.fit_att(panel_m, covariates=covs_m, response="burned")
    print("matched-DiD overall ATT:", overall_m)
except Exception as e:
    print("faithful-Castro event-time match / DiD skipped:", repr(e))

## 9 · Deconstruct `build_frame` — the tidy levels frame

The odds-ratio route consumes the matched `units` and rasterizes them onto the
grid: which cell belongs to which unit, its `treated`/`site_id` labels, the
covariates, and the per-year `burned` response. The subtle move — burning the
**carried** covariate values onto cells with `_rasterize_values` rather than
re-reading rasters at the zero-area point units.


🔎 **This cell:** reproduces `build_frame`'s labelling step: rasterize each unit's `unit_id`, `treated`, `site_id`, and carried covariate onto grid cells, building the `in_unit` mask of cells covered by a unit — showing values come from rasterizing carried columns, not re-reading rasters.

In [ ]:
units_ = units.to_crs(ANALYSIS_CRS).reset_index(drop=True)
grid = build_common_grid(units_, res_m=RES_M)

unit_ids = _rasterize_values(units_, units_["unit_id"], grid)
treated_r = _rasterize_values(units_, units_["treated"], grid)
site_ids = _rasterize_values(units_, units_["site_id"].astype("category").cat.codes, grid)
in_unit = ~np.isnan(unit_ids)                      # cells covered by SOME unit
print("grid cells:", unit_ids.size, "| cells inside a unit:", int(in_unit.sum()))

# a carried covariate, burned onto cells (NOT re-read from its raster)
name = covariates[0]
if name in units_.columns:
    burned_cov = _rasterize_values(units_, units_[name], grid)[in_unit]
    print(f"{name}: {np.isfinite(burned_cov).sum()} cells got a value via rasterize-carry")

🔎 **This cell:** calls the real `build_frame` to assemble the tidy pixel-year `frame` (the input to the logit) and prints the raw burn rate by treatment as a first descriptive signal.

In [ ]:
frame = build_frame(units, product="FireCCIS311", years=YEARS,
                    covariate_names=None, site_id_col="site_id")
print("frame shape:", frame.shape)
print("burn rate by treatment:")
print(frame.groupby("treated")["burned"].mean())
show(frame)

### Understanding check

🔎 **This cell:** quiz on why `build_frame` rasterizes carried covariate values instead of re-reading rasters.

In [ ]:
q = Quiz(
    prompt="""`build_frame` rasterizes the covariate VALUES it finds on `units` instead of
              calling covariate_on_grid() at the matched points. Why?""",
    options={
        "a": "Re-reading rasters is slower",
        "b": "Matched units are zero-area pixel-CENTROID points; clipping a raster to "
             "zero-area geometry nulls every cell — so it reuses the values matching "
             "already sampled, which also guarantees the model adjusts on exactly what "
             "the match balanced",
        "c": "The rasters were deleted after matching",
    },
    correct="b",
    explain="""
        A pixel centroid has no area. Clipping a raster mask to points masks everything,
        so re-deriving covariates would null the whole column. assemble_units deliberately
        carries the already-sampled covariate values through, and build_frame burns THOSE
        onto the cells — avoiding the trap and keeping match and model perfectly consistent.
    """,
)
q.ask()

🔎 **This cell:** your answer slot for the build_frame quiz.

In [ ]:
q.answer("?")

## 10 · Deconstruct `fit_logit_clustered` → `odds_ratios`

The headline model: logistic `burned ~ treated + covariates` with SEs **clustered
on site**. The pre-fit guardrails live in `_prepare_design`: build the formula,
drop NaN rows *ourselves* (so cluster groups stay aligned), center-and-scale the
continuous covariates for conditioning, and reject a rank-deficient design.


🔎 **This cell:** selects the adjustment covariates present in the frame and calls `_prepare_design` to show the built formula, the NaN-drop to complete cases, and the effective N (number of site clusters) the SEs will be based on.

In [ ]:
desired = ["elevation", "precip_normal", "tmax_normal", "gdd_normal",
           "soil_organic_matter", "soil_awc", "soil_site_index", "soil_water_table_depth"]
covs = [c for c in desired if c in frame.columns]
print("adjusting for:", covs)

# --- peek inside _prepare_design ---
clean, formula = _prepare_design(frame, covs, "burned", "treated", "site_id",
                                 formula=None, standardize=True)
print("formula:", formula)
print("rows before dropna:", len(frame), "-> complete-case rows:", len(clean))
print("site clusters (effective N):", clean["site_id"].nunique())

🔎 **This cell:** fits the cluster-robust logistic and prints the summary, then shows `odds_ratios` is literally `exp(beta)` (with exp'd CIs) by exponentiating the `treated` coefficient by hand.

In [ ]:
# --- the fit + the interpretable table ---
result = fit_logit_clustered(frame, covariates=covs)   # burned ~ treated + covs, SEs clustered on site
print(result.summary())

or_table = odds_ratios(result)
# odds_ratios is literally exp(beta) with exp(CI):
beta_treated = result.params["treated"]
print(f"\ntreated: beta={beta_treated:.3f}  ->  exp(beta) = odds ratio = {np.exp(beta_treated):.3f}")
from IPython.display import display
display(or_table)

### Understanding checks (the model)

🔎 **This cell:** quiz on why the SEs are clustered on site.

In [ ]:
q = Quiz(
    prompt="Why cluster the standard errors on `site_id` instead of a plain logit?",
    options={
        "a": "It makes the odds ratios larger",
        "b": "Pixels within one restoration site are spatially correlated, not independent; "
             "a plain logit treats each ~300 m pixel as its own observation and wildly "
             "overstates significance. Clustering makes the effective N the number of sites",
        "c": "statsmodels requires a cluster argument for logit",
    },
    correct="b",
    explain="""
        Thousands of correlated pixels are NOT thousands of independent data points. Naive
        SEs shrink toward zero and hand you false 'significance'. Clustering on the site
        inflates the SEs to reflect that the real replication is across the handful of
        restoration projects — the honest uncertainty.
    """,
)
q.ask()

🔎 **This cell:** your answer slot for the clustering quiz.

In [ ]:
q.answer("?")

🔎 **This cell:** quiz on whether standardizing changes the treated odds ratio.

In [ ]:
q = Quiz(
    prompt="`standardize=True` centers/scales the continuous covariates. Does that change "
           "the `treated` odds ratio?",
    options={
        "a": "Yes, it shrinks it toward 1",
        "b": "No — `treated` stays raw 0/1, so its odds ratio is unchanged. Standardizing "
             "is pure numerical conditioning; only each continuous covariate's OR becomes "
             "'per standard deviation'",
        "c": "Yes, it flips its sign",
    },
    correct="b",
    explain="""
        Standardizing fixes an ill-conditioned Hessian (elevation in metres vs precip in
        mm otherwise triggers 'Singular matrix'). It's a reparameterization of the
        continuous columns only; `treated` is left on its native 0/1 scale so the headline
        odds ratio is identical with or without it. Each scaled covariate just reports a
        per-SD odds ratio.
    """,
)
q.ask()

🔎 **This cell:** your answer slot for the standardize quiz.

In [ ]:
q.answer("?")

🔎 **This cell:** quiz on interpreting a `treated` odds ratio of 0.6 with CI [0.4, 0.9].

In [ ]:
q = Quiz(
    prompt="The `treated` odds ratio comes out at 0.6 with a 95% CI of [0.4, 0.9]. Read it.",
    options={
        "a": "Restoration RAISES the odds of a pixel-year burning by 60%",
        "b": "Restoration multiplies the odds of burning by ~0.6 (a ~40% reduction), and "
             "because the whole CI sits below 1 the protective effect is statistically clear",
        "c": "There is no detectable effect because 0.6 is close to 1",
    },
    correct="b",
    explain="""
        odds_ratio = exp(beta). Below 1 = protective (fewer fires). 0.6 means the odds of
        burning on a treated pixel-year are ~0.6x those on a comparable control — about a
        40% reduction. Fire is rare here, so the odds ratio ~ a risk ratio. The CI lying
        entirely below 1 is what makes it 'significant' — and it's only honest because the
        SEs were clustered on site.
    """,
)
q.ask()

🔎 **This cell:** your answer slot for the odds-ratio interpretation quiz.

In [ ]:
q.answer("?")

## 11 · Capstone — put the whole pipeline together

You've deconstructed every stage. Two final checks tie the two estimation routes
together.


🔎 **This cell:** capstone quiz 1 — order the whole levels/odds-ratio pipeline.

In [ ]:
q = Quiz(
    prompt="""Put the LEVELS / odds-ratio route in order:
              (1) fit_logit_clustered  (2) match_controls  (3) get_treated_and_control_pixels
              (4) build_frame  (5) attach_covariates  (6) assemble_units""",
    options={
        "a": "3 -> 5 -> 2 -> 6 -> 4 -> 1",
        "b": "2 -> 3 -> 5 -> 4 -> 6 -> 1",
        "c": "3 -> 2 -> 5 -> 1 -> 4 -> 6",
    },
    correct="a",
    explain="""
        Build the pixel(-year) panel (3), sample covariates onto it (5), match treated to
        control on those covariates (2), package the matched pixels into units (6), build
        the tidy pixel-year frame with the response (4), and finally fit the clustered
        logit and read odds ratios (1). Matching lives UPSTREAM of build_frame by design —
        the causal 'who is a control' decision is kept separate from frame assembly.
    """,
)
q.ask()

🔎 **This cell:** your answer slot for the pipeline-order quiz.

In [ ]:
q.answer("?")

🔎 **This cell:** capstone quiz 2 — the difference between the DiD and odds-ratio routes.

In [ ]:
q = Quiz(
    prompt="""This project ships TWO estimators on the same front half of the pipeline.
              What's the key difference between the DiD route (did.py) and the
              odds-ratio route (models.py)?""",
    options={
        "a": "They give identical numbers; one is just faster",
        "b": "The odds-ratio route compares LEVELS (adjusting for measured covariates) on "
             "a matched cross-section; the DiD route compares the CHANGE in burning after "
             "each site's restoration vs not-yet/never-treated controls, so it also "
             "removes TIME-INVARIANT unmeasured confounders",
        "c": "DiD needs no controls at all",
    },
    correct="b",
    explain="""
        Both start from the matched pixel panel. models.py fits burned ~ treated + covs and
        reads an odds ratio — honest only for the confounders you measured and matched on.
        did.py identifies off the before/after change relative to controls, so every
        persistent confounder you NEVER measured (drainage history, access, soil quirks)
        differences out. That's why Castro's design (match first, then staggered DiD) is
        the stronger one — the odds-ratio route is the interpretable cross-check.
    """,
)
q.ask()

🔎 **This cell:** your answer slot for the two-routes quiz.

In [ ]:
q.answer("?")

## Where to go next

* Re-run any deconstruction cell after changing a knob — `caliper`, `k`,
  `spillover_m`, the covariate list, `product=` — and watch the intermediates move.
* Swap `product="FireCCIS311"` for a severity layer in `attach_fire_response` /
  `build_frame` to model severity with the *same* machinery.
* Add a `treated:precip` or `treated:pdsi` interaction in `fit_logit_clustered`
  (`formula=...`) to test whether restoration helps more in dry/drought years.
* For the full math and design-decision log, see `modeling_notebook_explained.md`;
  for the runnable end-to-end version, `notebooks/modeling.ipynb`.
